In [18]:
GEMINI_1_5_FLASH_CONFIG = {
    "vocab_size": 256000,     
    "context_length": 1024, 
    "emb_dim": 256,            # Estimated hidden size (similar scale to Gemma 2 9B)
    "n_heads": 8,              # Query attention heads
    "n_kv_heads": 8, 
    "n_layers": 4,         
    "architecture_type": "dense_decoder", 
    "drop_rate": 0.0,
    "qkv_bias": False,
    "activation": "SwiGLU",
    "mlp_dim": 682,
}

[ Input Embedding Matrix: x ]
                             │
              ┌──────────────┴──────────────┐
              ▼                             ▼
          ( x * cos )                _rotate_half(x)
              │                             │
              │                             ▼
              │                     ( flipped_x * sin )
              │                             │
              └──────────────┬──────────────┘
                             ▼
                    [ Added Together ]
                             │
                             ▼
                 [ Rotated Token Embeddings ]

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RotatoryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, theta=10000.0):
        super().__init__()
        self.dim = dim # dim = head_dim
        #dim = 64, so generate even number series till 64
        inv_freq = 1.0/(theta ** (torch.arange(0, dim, 2).float() / dim))#theta_i = theta^(-2i/d)
        self.register_buffer("inv_freq", inv_freq, persistent=False)

        # Precompute text position indices
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        
        # Separate out absolute coordinate mappings for cosine and sine transformations
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)#emb.cos() = cos(m*theta_i)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def _rotate_half(self, x):
        x1 = x[..., :self.dim // 2]
        x2 = x[..., self.dim // 2:]
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, x, seq_len):
        # x shape: [batch_size, n_heads, seq_len, head_dim]
            cos = self.cos_cached[:seq_len, :].unsqueeze(0).unsqueeze(1) # [1, 1, seq_len, head_dim]
            sin = self.sin_cached[:seq_len, :].unsqueeze(0).unsqueeze(1)
        
        # Apply the rotation matrix calculation
            return (x * cos) + (self._rotate_half(x) * sin)

class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.emb_dim = cfg["emb_dim"]
        self.n_heads = cfg["n_heads"]
        self.n_kv_heads = cfg["n_kv_heads"]
        self.head_dim = self.emb_dim // self.n_heads

        self.num_queries_per_kv = self.n_heads // self.n_kv_heads

        self.q_proj = nn.Linear(self.emb_dim, self.n_heads * self.head_dim, bias=cfg["qkv_bias"])
        self.k_proj = nn.Linear(self.emb_dim, self.n_kv_heads * self.head_dim, bias=cfg["qkv_bias"])
        self.v_proj = nn.Linear(self.emb_dim, self.n_kv_heads * self.head_dim, bias=cfg["qkv_bias"])
        self.out_proj = nn.Linear(self.n_heads * self.head_dim, self.emb_dim, bias=False)

        self.rope = RotatoryEmbedding(dim=self.head_dim, max_seq_len=cfg["context_length"])

    def forward(self, x):
        b, s, c = x.shape
        q = self.q_proj(x).view(b, s, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, s, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, s, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = self.rope(q, seq_len=s)
        k = self.rope(k, seq_len=s)
        
        # Expand Key/Values to match Query groups if using strict GQA structures
        if self.num_queries_per_kv > 1:
            k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
            v = v.repeat_interleave(self.num_queries_per_kv, dim=1)

        mask = torch.triu(torch.full((s, s), float('-inf'), device=x.device), diagonal=1)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn_weights = attn_weights + mask.unsqueeze(0).unsqueeze(1)
        attn_weights = F.softmax(attn_weights, dim=-1)

        context = torch.matmul(attn_weights, v) # [b, n_heads, s, head_dim]
        context = context.transpose(1, 2).contiguous().view(b, s, -1)
        
        return self.out_proj(context)

class GeminiSwiGLU(nn.Module):
    """
    Modern 3-matrix gate architecture scaling out across calculated mlp dimensions.
    """
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg["emb_dim"], cfg["mlp_dim"], bias=False)
        self.w_up   = nn.Linear(cfg["emb_dim"], cfg["mlp_dim"], bias=False)
        self.w_down = nn.Linear(cfg["mlp_dim"], cfg["emb_dim"], bias=False)

    def forward(self, x):
        # Element-wise gating computation: Swish(Gate) * Up-Projection
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class DummyRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        x_normed = x * torch.rsqrt(variance + self.eps)
        return x_normed * self.weight

class DummyGeminiFlashBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Norm instances for incoming token routing
        self.attn_norm = DummyRMSNorm(cfg["emb_dim"])
        self.ffn_norm  = DummyRMSNorm(cfg["emb_dim"])
        
        # Sub-layer initializations
        self.attn = GQA(cfg)
        self.ffn  = GeminiSwiGLU(cfg)

    def forward(self, x):
        # GEMINI PARALLEL ROUTING EXECUTION:
        # Both computation streams branch simultaneously out from the structural input x.
        residual_attn = self.attn(self.attn_norm(x))
        residual_ffn  = self.ffn(self.ffn_norm(x))
        
        # Parallel skip-connection fusion execution point
        return x + residual_attn + residual_ffn

class DummyGemini15FlashModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # 1. Native Multimodal Tokenizer Embedding space
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        #No postional encoding due as it is directly implemented inside attention layer using RoPE(Rotatory posiitional encoding)

        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[DummyGeminiFlashBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = DummyRMSNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        x = self.tok_emb(in_idx) #create vector embeddings of tokens
        x = self.drop_emb(x) #dropout embedding layer to  intoduce variation in training patterns and reduce overfitting
        x = self.trf_blocks(x)#trnasformers block
        x = self.final_norm(x)#RMS Normalizaion layer
        logits = self.out_head(x)#Prediction Score for every single words
        return logits

In [6]:
from transformers import AutoTokenizer
from dotenv import load_dotenv
import os

load_dotenv()

# 2. Retrieve the token safely
hf_token = os.getenv("HF_ACCESS_TOKEN")

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b", token = hf_token)

In [8]:
batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[    2,  9112,  8395, 14574,   692],
        [    2,  9112,  1744, 12723,   476]])


In [21]:
torch.manual_seed(123)
model = DummyGemini15FlashModel(GEMINI_1_5_FLASH_CONFIG)
logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 5, 256000])
tensor([[[ 0.4045, -0.1929,  1.0951,  ...,  0.5795,  0.4946, -0.6040],
         [-0.2921, -0.1640, -0.0731,  ..., -0.3277, -0.2551,  0.3419],
         [ 0.3971,  0.3938, -0.5678,  ...,  0.1892, -0.9189,  1.4748],
         [ 0.5200, -0.4614,  0.7872,  ..., -0.2846,  0.3776,  1.2441],
         [ 0.9645,  0.1765, -1.2376,  ..., -0.1706, -0.3878,  0.6749]],

        [[ 0.4045, -0.1929,  1.0951,  ...,  0.5795,  0.4946, -0.6040],
         [-0.2921, -0.1640, -0.0731,  ..., -0.3277, -0.2551,  0.3419],
         [-0.0729, -0.6605, -0.2348,  ..., -0.2332, -0.5457, -1.1997],
         [-0.1351,  0.2707,  0.0474,  ...,  0.4101, -0.4209, -0.8386],
         [-0.7552,  0.2341, -0.2152,  ..., -0.1937, -0.7669,  1.3361]]],
       grad_fn=<UnsafeViewBackward0>)
